# Assignment 5
Develop an algorithm for finding the popular tweets on the twitter website based on the
concept of decaying window.

## Loading Dataset

In [1]:
import pandas as pd

twitter_file_path = '/kaggle/input/twitter-data-100x31/all_tweets.csv'
df = pd.read_csv(twitter_file_path)

print("Shape of the DataFrame: ", df.shape);
print("Columns of the DataFrame: ", df.columns);

Shape of the DataFrame:  (100, 31)
Columns of the DataFrame:  Index(['tweet_id', 'created_at', 'text', 'author_id', 'conversation_id',
       'in_reply_to_user_id', 'lang', 'possibly_sensitive', 'source',
       'like_count', 'retweet_count', 'reply_count', 'quote_count', 'username',
       'user_name', 'user_created_at', 'user_protected', 'user_verified',
       'media_type', 'media_url', 'media_duration_ms', 'media_width',
       'media_height', 'place_name', 'place_country', 'place_latitude',
       'place_longitude', 'poll_duration_minutes', 'poll_end_time',
       'poll_status', 'poll_options'],
      dtype='object')


#### Convert created_at to DateTime Format

In [3]:
df['created_at'] = pd.to_datetime(df['created_at'])

### Set the Current Time

In [5]:
from datetime import datetime

current_time = datetime.now()

current_time

datetime.datetime(2025, 3, 16, 7, 26, 48, 842551)

## Define the Decay Function

In [8]:
import numpy as np

λ = 0.01

def compute_decay(row):
    # Remove timezone info to make it naive
    t = (current_time - row['created_at'].tz_localize(None)).total_seconds() / 3600
    return np.exp(-λ * t)

### Compute the Popularity Score
Compute the weighted sum of the engagement metrics (like_count, retweet_count, reply_count, quote_count) using the decay factor:

Popularity Score
=(
like count
+
retweet count
+
reply count
+
quote count
)
×
𝑤
(
𝑡
)
Popularity Score=(like count+retweet count+reply count+quote count)×w(t)

In [9]:
def compute_popularity(row):
    decay = compute_decay(row)
    score = (row['like_count'] + row['retweet_count'] + row['reply_count'] + row['quote_count']) * decay
    return score

# Apply the function to compute popularity score
df['popularity_score'] = df.apply(compute_popularity, axis=1)

## Rank Tweets Based on Popularity

In [10]:
popular_tweets = df.sort_values(by='popularity_score', ascending=False)

## Display top 10 popular tweets

In [11]:
top_tweets = popular_tweets[['tweet_id', 'created_at', 'text', 'like_count', 'retweet_count', 'reply_count', 'quote_count', 'popularity_score']]

# Display the top 10 tweets
print(top_tweets.head(10))

               tweet_id                created_at  \
50  1900468334589620479 2025-03-14 08:45:34+00:00   
59  1900468108063547765 2025-03-14 08:44:40+00:00   
97  1900483133570527518 2025-03-14 09:44:22+00:00   
93  1900483630155145597 2025-03-14 09:46:21+00:00   
42  1900460162848350541 2025-03-14 08:13:06+00:00   
2   1900449800027271244 2025-03-14 07:31:55+00:00   
57  1900468186778067337 2025-03-14 08:44:59+00:00   
58  1900468143707111566 2025-03-14 08:44:49+00:00   
11  1900449891949646133 2025-03-14 07:32:17+00:00   
19  1900449344794615855 2025-03-14 07:30:07+00:00   

                                                 text  like_count  \
50  RT @OpenAI: OpenAI o1 and o3-mini now offer Py...           0   
59  RT @OpenAI: OpenAI o1 and o3-mini now offer Py...           0   
97  RT @Warcraft: A place in Azeroth of your very ...           0   
93  RT @chameleon_jeff: There's been a lot of disc...           0   
42  RT @chameleon_jeff: There's been a lot of disc...           0   
2 